<a href="https://www.kaggle.com/code/zainabhalhoul/finalctr?scriptVersionId=287686872" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import torch 
import torch.nn as nn
class ItemEmbeddingLayer(nn.Module):
    """
    e_item = concat( e_id(16d), e_tag1(32d), ..., e_tagT(32d), frozen_emb(128d) )
    Accepts frozen_mm (tensor) at init and uses it as lookup (not learnable).
    """
    def __init__(self, num_items, num_tag_ids, frozen_mm, tag_count_per_item=5):
        super().__init__()
        self.tag_count_per_item = tag_count_per_item
        self.frozen_mm = nn.Parameter(frozen_mm, requires_grad=False)  # [num_items, 128]
        self.id_emb = nn.Embedding(num_items, 64, padding_idx=0)      # 16-d
        self.tag_emb = nn.Embedding(num_tag_ids, 64, padding_idx=0)   # 32-d per tag

        # init (skip padding idx)
        if num_items > 1:
            nn.init.xavier_uniform_(self.id_emb.weight.data[1:])
        if num_tag_ids > 1:
            nn.init.xavier_uniform_(self.tag_emb.weight.data[1:])

        # ensure zero at padding
        with torch.no_grad():
            self.id_emb.weight.data[0].zero_()
            self.tag_emb.weight.data[0].zero_()

    def forward(self, item_ids, item_tags):
        """
        item_ids:  [B, N] or [N] or [B]             -- integer ids
        item_tags: [B, N, T] or [N, T] or [T]      -- tag ids per item (T tags)
        returns:   embeddings with shape [..., 16 + T*32 + 128]
        """
        # id embedding => [..., 16]
        id_vec = self.id_emb(item_ids)

        # tag embedding => [..., T, 32] then flatten to [..., T*32]
        tag_vecs = self.tag_emb(item_tags)   # tag ids of 0 -> zero vector
        # support broadcasting in case an input is [B, T] for single item:
        if tag_vecs.dim() == 2:  # [T, 32] or [B, T]? handle generically
            # If tag_vecs is [T,32] (single item case), make it [..., T, 32]
            tag_vecs = tag_vecs.unsqueeze(0)
        tag_flat = tag_vecs.view(*tag_vecs.shape[:-2], -1)  # [..., T*32]

        # frozen multimodal: lookup by item_ids (works with broadcasting)
        frozen_vec = self.frozen_mm[item_ids]  # => [..., 128]

        # concat final
        final = torch.cat([id_vec, tag_flat, frozen_vec], dim=-1)
        return final


# -------------------------
# Sequential Feature Learning Module (corrected)
# -------------------------
class SequentialFeatureLearning(nn.Module):
    """
    Implements Section 2.2.2 with corrections:
      - concatenates e_item_i || e_target per item (broadcasted)
      - Transformer with src_key_padding_mask
      - masked max-pool to ignore padded positions
      - handles k > N by zero-padding the short-term vector
    Inputs:
      history_ids  : [B, N]
      history_tags : [B, N, T]
      target_id    : [B]
      target_tags  : [B, T]
    Output:
      S_o shape = [B, k*dt + dt]  (stable even if k > N)
    """
    def __init__(self,
                 item_embedding_layer: ItemEmbeddingLayer,
                 embed_dim=512,   # item embedding dim
                 dt=128,
                 num_layers=2,
                 num_heads=4,
                 k=16,
                 dropout=0.2):
        super().__init__()
        self.item_embedding_layer = item_embedding_layer
        self.embed_dim = embed_dim            # 304
        self.concat_dim = embed_dim * 2       # e_item || e_target
        self.dt = dt
        self.k = k

        # project concatenated 2*304 -> dt
        self.input_proj = nn.Linear(self.concat_dim, dt)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dt,
            nhead=num_heads,
            batch_first=True,
            dim_feedforward=dt * 4,
            dropout=dropout,
            activation='relu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, history_ids, history_tags, target_id, target_tags):
        """
        history_ids:  [B, N]
        history_tags: [B, N, T]
        target_id:    [B]
        target_tags:  [B, T]
        """
        B, N = history_ids.shape
        device = history_ids.device

        # 1) get history embeddings: [B, N, 304]
        hist_emb = self.item_embedding_layer(history_ids, history_tags)   # [..., 304]

        # 2) get target embedding [B, 304] and expand to [B, N, 304] using unsqueeze+expand (proper broadcasting)
        targ_emb = self.item_embedding_layer(target_id, target_tags)     # [B, 304]
        targ_exp = targ_emb.unsqueeze(1).expand(-1, N, -1)               # [B, N, 304]

        # 3) concat per-item with target: [B, N, 608]
        seq = torch.cat([hist_emb, targ_exp], dim=-1)

        # 4) project to dt dim: [B, N, dt]
        seq = self.input_proj(seq)

        # 5) padding mask for transformer: True for positions that should be masked (id == 0)
        padding_mask = (history_ids == 0)   # bool tensor [B, N]

        # 6) transformer encoding (src_key_padding_mask expects shape [B, N], True = masked)
        # Note: TransformerEncoder with batch_first=True
        S = self.transformer(seq, src_key_padding_mask=padding_mask)   # [B, N, dt]

        # 7) Short-term: last k outputs.
        # If k > N, we use available outputs and left-pad with zeros so output shape is stable: [B, k*dt]
        k_eff = min(self.k, N)
        if k_eff > 0:
            last_k = S[:, -k_eff:, :]  # [B, k_eff, dt]
            last_k_flat = last_k.reshape(B, -1)  # [B, k_eff*dt]
            if k_eff < self.k:
                # Need to pad left with zeros to reach k*dt
                pad_elems = self.k - k_eff
                pad = torch.zeros(B, pad_elems * self.dt, device=device, dtype=last_k_flat.dtype)
                short_term = torch.cat([pad, last_k_flat], dim=-1)  # [B, k*dt]
            else:
                short_term = last_k_flat  # [B, k*dt]
        else:
            # k == 0 (edge case) => short_term zeros
            short_term = torch.zeros(B, self.k * self.dt, device=device, dtype=S.dtype)

        # 8) Long-term: masked max-pooling over time.
        # We must ignore padded positions (where padding_mask==True).
        # For masked positions set to a very small value before max.
        neg_inf = torch.tensor(-1e9, device=device, dtype=S.dtype)
        # expand padding mask to [B, N, 1] for broadcasting
        pm_expand = padding_mask.unsqueeze(-1)  # True where padding
        S_masked = torch.where(pm_expand, neg_inf, S)  # padded positions replaced by -inf
        long_term = torch.max(S_masked, dim=1).values  # [B, dt]

        # If all positions for a row are padding (all -inf), max returns -inf -> replace with zeros
        long_term = torch.where(torch.isfinite(long_term), long_term, torch.zeros_like(long_term))

        # 9) Final So: concat short_term (k*dt) and long_term (dt) => [B, k*dt + dt]
        S_o = torch.cat([short_term, long_term], dim=-1)
        return S_o
## side features embeddings
# Simplified version for your specific case:
class SimpleSideFeatureEmbedding(nn.Module):
    """
    Simplified version assuming like_level and view_level are categorical.
    """

    def __init__(self, like_vocab_size=11, view_vocab_size=11, emb_dim=16):
        super().__init__()

        self.like_emb = nn.Embedding(like_vocab_size, emb_dim, padding_idx=0)
        self.view_emb = nn.Embedding(view_vocab_size, emb_dim, padding_idx=0)

        self.output_dim = emb_dim * 2

        # Initialize
        nn.init.xavier_uniform_(self.like_emb.weight.data[1:])
        nn.init.xavier_uniform_(self.view_emb.weight.data[1:])

        with torch.no_grad():
            self.like_emb.weight.data[0].zero_()
            self.view_emb.weight.data[0].zero_()

    def forward(self, likes_level, views_level):
        e_like = self.like_emb(likes_level)  # [B, emb_dim]
        e_view = self.view_emb(views_level)  # [B, emb_dim]

        e_side = torch.cat([e_like, e_view], dim=-1)  # [B, output_dim]
        return e_side
#### deep cross network
import torch
import torch.nn as nn

class DCNv2FeatureInteraction(nn.Module):
    """
    DCNv2 feature interaction module (Section 2.2.3, Equations 7-8).

    Inputs:
        e_target : [B, D_t]   # Target item embedding
        e_side   : [B, D_s]   # Side feature embeddings
        S_o      : [B, D_sqo] # Sequential features from transformer

    Output:
        f_o : [B, input_dim + deep_output_dim] = [c_o, d_o]
    """
    def __init__(
        self,
        input_dim,                  # D = D_t + D_s + D_sqo
        num_cross_layers=3,
        deep_hidden_dims=[1024, 512, 256], # Table 1
        deep_output_dim=256,        # d_o dimension
        dropout_rate=0.2
    ):
        super().__init__()
        self.input_dim = input_dim
        self.num_cross_layers = num_cross_layers

        # Cross Network (c_{l+1} = f_i ⊙ (W_l c_l + b_l) + c_l)
        self.cross_W = nn.ModuleList([nn.Linear(input_dim, input_dim, bias=False)
                                      for _ in range(num_cross_layers)])
        self.cross_b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim))
                                         for _ in range(num_cross_layers)])

        # Deep Network: MLP_f(f_i)
        deep_layers = []
        dim = input_dim
        for h in deep_hidden_dims:
            deep_layers.append(nn.Linear(dim, h))
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.Dropout(dropout_rate))
            dim = h
        deep_layers.append(nn.Linear(dim, deep_output_dim))
        self.deep_mlp = nn.Sequential(*deep_layers)

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, e_target, e_side, S_o):
        # Initial concatenation: f_i = [e_target || e_side || S_o]
        f_i = torch.cat([e_target, e_side, S_o], dim=-1)

        # Cross Network
        c_l = f_i
        for l in range(self.num_cross_layers):
            linear_out = self.cross_W[l](c_l) + self.cross_b[l]
            c_l = f_i * linear_out + c_l
            c_l = self.dropout(c_l)
        c_o = c_l

        # Deep Network
        d_o = self.deep_mlp(self.dropout(f_i))

        # Parallel output
        f_o = torch.cat([c_o, d_o], dim=-1)
        return f_o
### prediction layer & loos function
import torch.nn.functional as F
import polars as pl

class CTRPredictionLayer(nn.Module):
    """
    Prediction layer (Section 2.2.4).
    2-layer perceptron with hidden units [64, 32] and sigmoid output.
    """
    def __init__(self, input_dim, hidden_dims=[64, 32], dropout_rate=0.0):
        super().__init__()
        assert len(hidden_dims) == 2, "Should be a 2-layer perceptron"

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(),
            #nn.Dropout(dropout_rate),

            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            #nn.Dropout(dropout_rate),

            nn.Linear(hidden_dims[1], 1)  # Output layer
        )

        self._init_weights()

    def _init_weights(self):
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)

    def forward(self, f_o):
        logits = self.mlp(f_o)
        y_hat= torch.sigmoid(logits)
        return y_hat



class CTRLoss(nn.Module):
    """
    Binary cross-entropy loss (Section 2.2.5).
    """
    def __init__(self, reduction='mean'):
        super().__init__()
        self.reduction = reduction

    def forward(self, y_hat, y):
        if y.dim() == 1:
            y = y.unsqueeze(-1)
        loss = F.binary_cross_entropy(y_hat, y, reduction=self.reduction)
        return loss
### dataloader
from logging import INFO
import numpy as np
import polars as pl
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Optional

# ===============================
# MMCTR Dataset
# ===============================
class MMCTRDataset(Dataset):
    """
    Dataset for WWW 2025 MM-CTR Challenge.
    Loads and preprocesses train/validation/test parquet files.
    """

    def __init__(self, data_path: str, item_info_path: str, max_seq_len: int = 64):
        self.max_seq_len = max_seq_len

        # Load data
        self.data_df = pl.read_parquet(data_path).to_pandas()
        self.item_info_df = pl.read_parquet(item_info_path).to_pandas()

        # Create mapping: item_id -> tags
        self.item_tags_dict = dict(zip(
            self.item_info_df['item_id'],
            self.item_info_df['item_tags']
        ))
        self.item_info_df = self.item_info_df.sort_values('item_id')
        emb_list = self.item_info_df['item_emb_d128'].tolist()
        self.frozen_embeddings = torch.tensor(emb_list, dtype=torch.float32)
       

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]

        # -----------------------------
        # 1. History sequence (item IDs)
        # -----------------------------
        history_ids = row['item_seq']
        history_ids = self._pad_or_truncate(history_ids, self.max_seq_len, pad_value=0)

        # -----------------------------
        # 2. History tags
        # -----------------------------
        history_tags = [self._get_item_tags(item_id) for item_id in history_ids]

        # -----------------------------
        # 3. Target item
        # -----------------------------
        target_id = row['item_id']
        target_tags = self._get_item_tags(target_id)

        # -----------------------------
        # 4. Side features
        # -----------------------------
        likes_level = row.get('likes_level', 0)
        views_level = row.get('views_level', 0)

        # -----------------------------
        # 5. Label
        # -----------------------------
        label = row.get('label', 0.0)

        # -----------------------------
        # 6. Convert to tensors
        # -----------------------------
        sample = {
            'history_ids': torch.tensor(history_ids, dtype=torch.long),
            # TEMPORARY: Go back to old working version!
            'history_tags': torch.tensor(history_tags, dtype=torch.long),
            'target_id': torch.tensor(target_id, dtype=torch.long),
            'target_tags': torch.tensor(target_tags, dtype=torch.long),
            'likes_level': torch.tensor(likes_level, dtype=torch.long),
            'views_level': torch.tensor(views_level, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float)
        }
        return sample

    # ===============================
    # Helper functions
    # ===============================
    def _get_item_tags(self, item_id: int) -> List[int]:
        if item_id == 0:
            return [0] * 5
        
        tags = self.item_tags_dict.get(item_id, [0, 0, 0, 0, 0])
        
        # Ensure exactly 5 tags
        if len(tags) < 5:
            tags = tags + [0] * (5 - len(tags))
        elif len(tags) > 5:
            tags = tags[:5]
        
        return tags

    def _pad_or_truncate(self, seq: List[int], max_len: int, pad_value: int = 0) -> List[int]:
        if not isinstance(seq, list):
            seq = list(seq)
        
        if len(seq) > max_len:
            return seq[-max_len:]
        elif len(seq) < max_len:
            return seq + [pad_value] * (max_len - len(seq))
        else:
            return seq


# ===============================
# MMCTR Collator
# ===============================
class MMCTRCollator:
    """
    Collator to batch samples together for DataLoader.
    """

    def __call__(self, batch: List[Dict]) -> Dict[str, torch.Tensor]:
        batched = {}
        for key in batch[0].keys():
            tensors = [sample[key] for sample in batch]

            if key in ['history_ids', 'history_tags', 'target_tags']:
                batched[key] = torch.stack(tensors, dim=0)
            elif key == 'label':
                batched[key] = torch.stack(tensors, dim=0).unsqueeze(-1)
            else:
                batched[key] = torch.stack(tensors, dim=0)

        return batched


# ===============================
# Create DataLoaders
# ===============================
def create_dataloaders(
    train_path: str,
    val_path: Optional[str] = None,
    test_path: Optional[str] = None,
    item_info_path: str = "/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/MicroLens_1M_x1/item_info.parquet",
    batch_size: int = 128,
    max_seq_len: int = 64,
    num_workers: int = 4,
    shuffle_train: bool = True
):
    """
    Creates DataLoaders for train, validation, and test sets.
    """
    dataloaders = {}
    collator = MMCTRCollator()

    # Train
    train_dataset = MMCTRDataset(train_path, item_info_path, max_seq_len)
    dataloaders['train'] = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        collate_fn=collator,
        pin_memory=True
    )

    # Validation
    if val_path:
        val_dataset = MMCTRDataset(val_path, item_info_path, max_seq_len)
        dataloaders['val'] = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            collate_fn=collator,
            pin_memory=True
        )

    # Test
    if test_path:
        test_dataset = MMCTRDataset(test_path, item_info_path, max_seq_len)
        dataloaders['test'] = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            collate_fn=collator,
            pin_memory=True
        )

    # Statistics
    print("\n" + "="*60)
    print("DataLoader Statistics")
    print("="*60)
    print(f"Train samples: {len(train_dataset)}")
    if val_path:
        print(f"Validation samples: {len(val_dataset)}")
    if test_path:
        print(f"Test samples: {len(test_dataset)}")
    print(f"Batch size: {batch_size}")
    print(f"Max sequence length: {max_seq_len}")

    # Sample batch shapes
    sample_batch = next(iter(dataloaders['train']))
    print("\nSample batch shapes:")
    for key, value in sample_batch.items():
        print(f"  {key}: {value.shape} ({value.dtype})")

    return dataloaders
### full model
import torch
import torch.nn as nn
import polars as pl
import numpy as np

 


# ===============================
# COMPLETE WWW 2025 MODEL - FIXED
# ===============================
class CompleteWWW2025Model(nn.Module):


    
    def __init__(
        self,
        # Item embedding parameters
        num_items,
        num_tags,
        frozen_multimodal_embeddings,  # Keep this name
        tag_count_per_item=5,
        item_id_emb_dim=16,  # FIXED: Should be 16 (matching ItemEmbeddingLayer)
        tag_emb_dim=32,      # FIXED: Should be 32 (matching ItemEmbeddingLayer)
        multimodal_emb_dim=128,

        # Sequential module parameters
        seq_num_layers=2,
        seq_num_heads=4,
        seq_dt=128,
        seq_k=16,
        seq_dropout=0.2,

        # Side feature parameters
        like_vocab_size=11,
        view_vocab_size=11,
        side_emb_dim=16,

        # DCNv2 parameters
        dcn_num_layers=3,
        dcn_deep_hidden_dims=[1024, 512, 256],
        dcn_deep_output_dim=256,
        dcn_dropout=0.2,

        # Prediction parameters
        pred_hidden_dims=[64, 32],
        pred_dropout=0.0
    ):
        super().__init__()

        print("="*60)
        print("Initializing Complete WWW 2025 Model")
        print("="*60)

        # ------------------------------
        # 1. ITEM EMBEDDING LAYER - FIXED
        # ------------------------------
        self.item_embedding = ItemEmbeddingLayer(
            num_items=num_items,
            num_tag_ids=num_tags,
            frozen_mm=frozen_multimodal_embeddings,  # FIXED: Parameter name
            tag_count_per_item=tag_count_per_item
        )
        self.item_emb_dim = item_id_emb_dim + tag_emb_dim * tag_count_per_item + multimodal_emb_dim
        print(f"Item embedding dimension: {self.item_emb_dim}")

        # ------------------------------
        # 2. SEQUENTIAL FEATURE LEARNING
        # ------------------------------
        self.sequential_module = SequentialFeatureLearning(
            item_embedding_layer=self.item_embedding,
            embed_dim=self.item_emb_dim,
            dt=seq_dt,
            num_layers=seq_num_layers,
            num_heads=seq_num_heads,
            k=seq_k,
            dropout=seq_dropout
        )
        self.S_o_dim = seq_k * seq_dt + seq_dt
        print(f"Sequential output dimension (S_o): {self.S_o_dim}")

        # ------------------------------
        # 3. SIDE FEATURE EMBEDDING
        # ------------------------------
        self.side_embedding = SimpleSideFeatureEmbedding(
            like_vocab_size=like_vocab_size,
            view_vocab_size=view_vocab_size,
            emb_dim=side_emb_dim
        )
        self.side_dim = side_emb_dim * 2
        print(f"Side feature dimension: {self.side_dim}")

        # ------------------------------
        # 4. DCNv2 FEATURE INTERACTION
        # ------------------------------
        dcn_input_dim = self.item_emb_dim + self.side_dim + self.S_o_dim
        print(f"DCNv2 input dimension: {dcn_input_dim}")

        self.feature_interaction = DCNv2FeatureInteraction(
            input_dim=dcn_input_dim,
            num_cross_layers=dcn_num_layers,
            deep_hidden_dims=dcn_deep_hidden_dims,
            deep_output_dim=dcn_deep_output_dim,
            dropout_rate=dcn_dropout
        )

        # ------------------------------
        # 5. PREDICTION LAYER
        # ------------------------------
        pred_input_dim = dcn_input_dim + dcn_deep_output_dim
        print(f"Prediction layer input dimension: {pred_input_dim}")

        self.prediction_layer = CTRPredictionLayer(
            input_dim=pred_input_dim,
            hidden_dims=pred_hidden_dims,
            dropout_rate=pred_dropout
        )

        # ------------------------------
        # 6. LOSS FUNCTION
        # ------------------------------
        self.loss_fn = CTRLoss(reduction='mean')
        
        print("="*60)
        print("Model initialization complete!")
        print("="*60)

    # ===============================
    # Forward Pass - UNCHANGED (it's correct)
    # ===============================
    def forward(self, batch_data, compute_loss=False):
        """
        Forward pass through the complete model.
        
        Args:
            batch_data: dict containing:
                - history_ids: [B, N]
                - history_tags: [B, N, 5]
                - target_id: [B]
                - target_tags: [B, 5]
                - likes_level: [B]
                - views_level: [B]
                - label: [B, 1] (optional, for training)
            
            compute_loss: if True and label provided, compute loss
        
        Returns:
            y_hat: [B, 1] predicted CTR probability
            loss: scalar (if compute_loss=True and label provided)
        """
        # Debug: Check input keys
        expected_keys = {'history_ids', 'history_tags', 'target_id', 'target_tags', 
                        'likes_level', 'views_level'}
        
        missing_keys = expected_keys - set(batch_data.keys())
        if missing_keys:
            print(f"Warning: Missing keys in batch_data: {missing_keys}")
        
        # 1. Target item embedding
        target_emb = self.item_embedding(
            batch_data['target_id'], 
            batch_data['target_tags']
        )  # [B, item_emb_dim]

        # 2. Sequential features (Transformer)
        S_o = self.sequential_module(
            batch_data['history_ids'],
            batch_data['history_tags'],
            batch_data['target_id'],
            batch_data['target_tags']
        )  # [B, S_o_dim]

        # 3. Side feature embeddings
        e_side = self.side_embedding(
            batch_data['likes_level'],
            batch_data['views_level']
        )  # [B, side_dim]

        # 4. DCNv2 feature interaction
        f_o = self.feature_interaction(target_emb, e_side, S_o)  # [B, dcn_input_dim + dcn_deep_output_dim]

        # 5. Prediction - THIS IS THE FINAL OUTPUT!
        y_hat = self.prediction_layer(f_o)  # [B, 1] predicted CTR probability

        # 6. Compute loss if requested
        loss = None
        if compute_loss and 'label' in batch_data:
            # Ensure label has correct shape
            label = batch_data['label']
            if label.dim() == 1:
                label = label.unsqueeze(-1)  # [B] -> [B, 1]
            
            # Debug: Check shapes match
            if y_hat.shape != label.shape:
                print(f"Warning: Shape mismatch - y_hat: {y_hat.shape}, label: {label.shape}")
            
            loss = self.loss_fn(y_hat, label)

        return y_hat, loss

    # ===============================
    # Inference
    # ===============================
    def predict(self, batch_data):
        """Inference mode forward pass."""
        with torch.no_grad():
            y_hat, _ = self.forward(batch_data, compute_loss=False)
            return y_hat


# ===============================
# NEW: Create model from dataset (not from file)
# ===============================
def create_complete_model_from_dataset(dataset, device=None):
    """
    Create model using frozen embeddings from an existing dataset.
    This avoids duplicate loading.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\nCreating model on device: {device}")
    
    # Get frozen embeddings from dataset
    frozen_emb_tensor = dataset.frozen_embeddings.to(device)
    frozen_emb_tensor.requires_grad = False
    
    # Get dimensions from dataset
    num_items = len(frozen_emb_tensor)
    
    # Get unique tags from dataset
    all_tags = []
    for tags in dataset.item_tags_dict.values():
        all_tags.extend(tags)
    
    num_tags = max(all_tags) + 1 if all_tags else 1
    
    print(f"Dataset info: {num_items} items, {num_tags} unique tags")
    
    # Define parameters with CORRECT dimensions
    item_id_emb_dim = 64  # From your ItemEmbeddingLayer
    tag_emb_dim = 64      # From your ItemEmbeddingLayer
    multimodal_emb_dim = 128
    tag_count_per_item = 5
    
    # Calculate actual item embedding dimension
    actual_item_emb_dim = item_id_emb_dim + (tag_emb_dim * tag_count_per_item) + multimodal_emb_dim
    print(f"Actual item embedding dimension: {actual_item_emb_dim}")
    
    # Create model with CORRECT dimensions
    model = CompleteWWW2025Model(
        num_items=num_items,
        num_tags=num_tags,
        frozen_multimodal_embeddings=frozen_emb_tensor,
        tag_count_per_item=tag_count_per_item,
        item_id_emb_dim=item_id_emb_dim,  # 16
        tag_emb_dim=tag_emb_dim,          # 32
        multimodal_emb_dim=multimodal_emb_dim,
        seq_num_layers=2,
        seq_num_heads=2,
        seq_dt=128,
        seq_k=16,
        seq_dropout=0.1,
        like_vocab_size=11,
        view_vocab_size=11,
        side_emb_dim=32,
        dcn_num_layers=3,
        dcn_deep_hidden_dims=[1024, 512, 256],
        dcn_deep_output_dim=256,
        dcn_dropout=0.1,
        pred_hidden_dims=[64, 32],
        pred_dropout=0.0
    ).to(device)
    
    return model
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from tqdm import tqdm
import time

# ===============================
# Training function - FIXED VERSION
# ===============================
def _get_gradient_norm(model):
    """Helper to get gradient norm for monitoring."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    return f"{total_norm:.2f}"

def train_model(model, dataloaders, epochs=50, lr=5e-4,  # Changed epochs to 50 (paper uses early stopping)
                patience=5, clip_grad=1.0, device=None,  # Reduced clip_grad
                weight_decay=1e-5, pos_weight=None):     # Removed default pos_weight
    """
    Train model according to paper specifications.
    """
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    # ============================================
    # 1. FIXED: Use BCELoss (not BCEWithLogitsLoss) since prediction layer has sigmoid
    # ============================================
    if pos_weight is not None:
        pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float).to(device)
        print(f"Using class weighting: pos_weight={pos_weight:.1f}")
        # BCELoss for weighted binary cross-entropy (since we have sigmoid output)
        loss_fn = nn.BCELoss(weight=pos_weight_tensor, reduction='mean')
    else:
        print("No class weighting applied")
        # Use model's built-in loss function (CTRLoss which is BCELoss)
        loss_fn = model.loss_fn
    
    # ============================================
    # 2. OPTIMIZER with L2 REGULARIZATION (paper uses Adam)
    # ============================================
    optimizer = optim.Adam(model.parameters(), 
                          lr=lr, 
                          betas=(0.9, 0.999),
                          weight_decay=weight_decay)
    
    print(f"Using L2 regularization: weight_decay={weight_decay}")
    
    # ============================================
    # 3. LEARNING RATE SCHEDULER (paper doesn't mention scheduler, but useful)
    # ============================================
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-6
    )

    # ============================================
    # 4. TRAINING HISTORY
    # ============================================
    history = {
        'train_loss': [], 'train_auc': [], 'train_logloss': [], 'train_acc': [],
        'val_loss': [], 'val_auc': [], 'val_logloss': [], 'val_acc': [],
        'best_val_auc': 0.0, 'epochs_no_improve': 0, 'best_epoch': 0,
        'learning_rates': []
    }

    print(f"\nTraining Configuration:")
    print(f"  Device: {device}")
    print(f"  Train samples: {len(dataloaders['train'].dataset):,}")
    print(f"  Val samples: {len(dataloaders['val'].dataset):,}")
    print(f"  Batch size: {dataloaders['train'].batch_size}")
    print(f"  Learning rate: {lr} (paper: 5e-4)")
    print(f"  Gradient clipping: {clip_grad}")
    print(f"  Weight decay (L2): {weight_decay}")
    print(f"  Early stopping patience: {patience} (paper: 5)")

    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch + 1}/{epochs}")
        print(f"{'='*60}")

        start_time = time.time()

        # ============================
        # TRAINING
        # ============================
        model.train()
        train_loss = 0.0
        y_true_train, y_pred_train, y_prob_train = [], [], []

        train_bar = tqdm(dataloaders['train'], desc=f"Training Epoch {epoch+1}")
        for batch_idx, batch in enumerate(train_bar):
            # Move to device
            batch = {k: v.to(device) for k, v in batch.items()}

            # Forward pass - FIXED: Use model's forward but compute loss separately
            y_hat, _ = model(batch, compute_loss=False)  # Get predictions
            
            # Compute loss using the appropriate loss function
            labels = batch['label']
            if labels.dim() == 1:
                labels = labels.unsqueeze(-1)
            loss = loss_fn(y_hat, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
            if clip_grad > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

            optimizer.step()

            # Accumulate stats
            train_loss += loss.item() * batch['label'].size(0)

            # Store predictions
            y_true_train.extend(batch['label'].cpu().numpy().flatten())
            y_prob_train.extend(y_hat.detach().cpu().numpy().flatten())
            y_pred_train.extend((y_hat.detach().cpu().numpy() > 0.5).astype(int).flatten())

            # Update progress bar
            train_bar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'lr': f"{optimizer.param_groups[0]['lr']:.6f}",
                'grad_norm': _get_gradient_norm(model)})

        # Calculate training metrics
        train_loss /= len(dataloaders['train'].dataset)
        train_auc = roc_auc_score(y_true_train, y_prob_train)
        train_logloss = log_loss(y_true_train, y_prob_train)
        train_acc = accuracy_score(y_true_train, y_pred_train)

        history['train_loss'].append(train_loss)
        history['train_auc'].append(train_auc)
        history['train_logloss'].append(train_logloss)
        history['train_acc'].append(train_acc)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])

        print(f"\n TRAIN SUMMARY:")
        print(f"  Loss:     {train_loss:.6f}")
        print(f"  AUC:      {train_auc:.6f}")
        print(f"  LogLoss:  {train_logloss:.6f}")
        print(f"  Accuracy: {train_acc:.4f}")

        # ============================
        # VALIDATION
        # ============================
        if 'val' in dataloaders:
            model.eval()
            val_loss = 0.0
            y_true_val, y_pred_val, y_prob_val = [], [], []

            with torch.no_grad():
                val_bar = tqdm(dataloaders['val'], desc=f"Validation Epoch {epoch+1}")
                for batch in val_bar:
                    batch = {k: v.to(device) for k, v in batch.items()}

                    y_hat, _ = model(batch, compute_loss=False)
                    
                    # Compute validation loss
                    labels = batch['label']
                    if labels.dim() == 1:
                        labels = labels.unsqueeze(-1)
                    loss = loss_fn(y_hat, labels)

                    val_loss += loss.item() * batch['label'].size(0)

                    y_true_val.extend(batch['label'].cpu().numpy().flatten())
                    y_prob_val.extend(y_hat.cpu().numpy().flatten())
                    y_pred_val.extend((y_hat.cpu().numpy() > 0.5).astype(int).flatten())

            # Calculate validation metrics
            val_loss /= len(dataloaders['val'].dataset)
            val_auc = roc_auc_score(y_true_val, y_prob_val)
            val_logloss = log_loss(y_true_val, y_prob_val)
            val_acc = accuracy_score(y_true_val, y_pred_val)

            history['val_loss'].append(val_loss)
            history['val_auc'].append(val_auc)
            history['val_logloss'].append(val_logloss)
            history['val_acc'].append(val_acc)

            print(f"\n VALIDATION SUMMARY:")
            print(f"  Loss:     {val_loss:.6f}")
            print(f"  AUC:      {val_auc:.6f}")
            print(f"  LogLoss:  {val_logloss:.6f}")
            print(f"  Accuracy: {val_acc:.4f}")
            
            # Calculate gap metrics
            auc_gap = train_auc - val_auc
            loss_gap = val_loss - train_loss
            print(f"  Gap - AUC: {auc_gap:.4f}, Loss: {loss_gap:.4f}")

            # ============================================
            # 6. LEARNING RATE SCHEDULING
            # ============================================
            scheduler.step(val_auc)
            
            # Check if learning rate changed
            current_lr = optimizer.param_groups[0]['lr']
            if epoch > 0 and current_lr < history['learning_rates'][-2]:
                print(f"   Learning rate reduced to: {current_lr:.6f}")

            # ============================================
            # 7. EARLY STOPPING (paper: stop if no improvement for 5 epochs)
            # ============================================
            if val_auc > history['best_val_auc']:
                history['best_val_auc'] = val_auc
                history['best_epoch'] = epoch + 1
                history['epochs_no_improve'] = 0

                # Save best model
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_auc': val_auc,
                    'val_loss': val_loss,
                    'train_auc': train_auc,
                    'train_loss': train_loss,
                }, 'best_model.pth')
                print(f"  ✓ Saved best model (AUC: {val_auc:.6f})")
            else:
                history['epochs_no_improve'] += 1
                print(f"   No improvement for {history['epochs_no_improve']}/{patience} epochs")

                if history['epochs_no_improve'] >= patience:
                    print(f"\n  ⏹️ Early stopping at epoch {epoch + 1}")
                    print(f"  Best validation AUC: {history['best_val_auc']:.6f} at epoch {history['best_epoch']}")
                    break

        epoch_time = time.time() - start_time
        print(f"\n⏱️  Epoch time: {epoch_time:.2f}s")

    return history

# ===============================
# Updated Evaluation function
# ===============================
def evaluate_model(model, dataloader, device=None):
    """Evaluate model with detailed metrics."""
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    all_predictions = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            batch = {k: v.to(device) for k, v in batch.items()}

            y_hat, _ = model(batch, compute_loss=False)
            
            # Compute loss
            labels = batch['label']
            if labels.dim() == 1:
                labels = labels.unsqueeze(-1)
            loss = model.loss_fn(y_hat, labels)

            total_loss += loss.item() * batch['label'].size(0)
            all_predictions.extend(y_hat.cpu().numpy().flatten())
            all_labels.extend(batch['label'].cpu().numpy().flatten())

    # Convert to numpy
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    avg_loss = total_loss / len(dataloader.dataset)
    auc = roc_auc_score(all_labels, all_predictions)
    logloss = log_loss(all_labels, all_predictions)
    
    binary_preds = (all_predictions > 0.5).astype(int)
    acc = accuracy_score(all_labels, binary_preds)

    return {
        'loss': avg_loss,
        'auc': auc,
        'logloss': logloss,
        'accuracy': acc,
        'predictions': all_predictions,
        'labels': all_labels
    }

# ===============================
# Main training script - FIXED
# ===============================
if __name__ == "__main__":
    import sys
    import os
    
    # Import your modules
    sys.path.append('.')  # Add current directory to path
    
    # FIXED: Import from your dataloader and model files
   # Use the new function
    
    # Paths (update these)
    TRAIN_PATH = "/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/MicroLens_1M_x1/train.parquet"
    VAL_PATH = "/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/MicroLens_1M_x1/valid.parquet"
    TEST_PATH = "/kaggle/input/www2025-mmctr-data/MicroLens_1M_MMCTR/MicroLens_1M_x1/test.parquet"
    ITEM_INFO_PATH = "/kaggle/input/task1-output/new_item_info.parquet"

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # ============================
    # 1. Create DataLoaders
    # ============================
    print("\nCreating DataLoaders...")
    dataloaders = create_dataloaders(
        train_path=TRAIN_PATH,
        val_path=VAL_PATH,
        test_path=TEST_PATH,
        item_info_path=ITEM_INFO_PATH,
        batch_size=128,  # FIXED: Paper uses 128
        max_seq_len=64,   # You can adjust this
        num_workers=4,
        shuffle_train=True
    )

    # ============================
    # 2. Create Model using dataset's frozen embeddings
    # ============================
    print("\nCreating model...")
    # Get dataset from train loader
    train_dataset = dataloaders['train'].dataset
    
    # Use the new function that takes dataset (not file path)
   
    model = create_complete_model_from_dataset(train_dataset, device=device)

    # Print model summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {total_params - trainable_params:,}")

    # ============================
    # 3. LAUNCH TRAINING
    # ============================
    print("\n" + "="*60)
    print("STARTING TRAINING")
    print("="*60)
    
    history = train_model(
        model=model,
        dataloaders=dataloaders,
        epochs=15,           # Paper uses early stopping
        lr=5e-4,            # Paper: 5e-4
        patience=3,         # Paper: stop if no improvement for 5 epochs
        clip_grad=3.0,      # Reasonable gradient clipping
        device=device,
        weight_decay=1e-5,  # L2 regularization
        pos_weight=3.004     # No class weighting unless your data is imbalanced
    )
    
    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"Best validation AUC: {history['best_val_auc']:.6f}")
    print(f"Best epoch: {history['best_epoch']}")


Using device: cuda

Creating DataLoaders...


/tmp/ipykernel_55/3755451989.py:339: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.frozen_embeddings = torch.tensor(emb_list, dtype=torch.float32)



DataLoader Statistics
Train samples: 3600000
Validation samples: 10000
Test samples: 379142
Batch size: 128
Max sequence length: 64

Sample batch shapes:
  history_ids: torch.Size([128, 64]) (torch.int64)
  history_tags: torch.Size([128, 64, 5]) (torch.int64)
  target_id: torch.Size([128]) (torch.int64)
  target_tags: torch.Size([128, 5]) (torch.int64)
  likes_level: torch.Size([128]) (torch.int64)
  views_level: torch.Size([128]) (torch.int64)
  label: torch.Size([128, 1]) (torch.float32)

Creating model...

Creating model on device: cuda
Dataset info: 91718 items, 11740 unique tags
Actual item embedding dimension: 512
Initializing Complete WWW 2025 Model
Item embedding dimension: 512
Sequential output dimension (S_o): 2176
Side feature dimension: 64
DCNv2 input dimension: 2752
Prediction layer input dimension: 3008
Model initialization complete!
Total parameters: 45,354,113
Trainable parameters: 33,614,209
Frozen parameters: 11,739,904

STARTING TRAINING
Using class weighting: pos_w

Training Epoch 1:  39%|███▉      | 10944/28125 [05:35<08:41, 32.97it/s, loss=0.3477, lr=0.000500, grad_norm=1.71]